# Lecture 6.1: How Sessions Work: Transcripts on Your Filesystem

Every time an agent runs, it builds up a record of what happened. That record is called a **session**. In this notebook you will run a real agent, then go and look at its session on disk, read it back with the SDK, and prove that a brand-new `query()` call knows nothing about it.

By the end of this notebook you will have seen six ideas with your own eyes:

1. A session is the saved conversation: your prompt, every tool call, every tool result and every response, in order. It exists independently of the files the agent worked on.
2. A `query()` call produces one session, and its ID arrives at the end of the stream on the final `ResultMessage`.
3. The SDK writes the session to disk automatically as a transcript (a `.jsonl` file) under a folder derived from the working directory.
4. The raw transcript is internal. We will look at it to make the idea concrete, but we read sessions with SDK functions.
5. A session stores the conversation, not your files.
6. A new `query()` call starts a new session with no memory of the previous one.

The scenario is a community garden's **Seed Library**: a couple of small text files that the agent will read.

## Step 1: Install the Claude Agent SDK

This cell installs `claude-agent-sdk`, the Python package that lets you run Claude as an agent from your own code. It is the only package this notebook needs.

| Part of the command | What it does |
| :- | :- |
| `!pip install` | Runs pip, Python's package installer, from inside the notebook |
| `claude-agent-sdk==0.2.167` | Installs exactly this version, so the notebook behaves the way it does in the video |
| `-q` | Quiet mode, so you see a short output instead of a long install log |

The comment block in the cell explains why the version is pinned. If this exact version is already installed in your Colab session, pip notices and finishes almost instantly. Run the cell and wait for it to finish before moving on.

In [1]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.167 -q

## Step 2: Set up your API key

The agent needs your Anthropic API key to talk to Claude. In Colab, the safe way to provide it is **Colab Secrets**, which keeps the key out of your notebook cells.

**Add the secret (one time):**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your API key as the value. You create keys in the Claude Console at https://platform.claude.com/ under **API Keys**.
5. Switch **Notebook access** on for this notebook.

The cell below reads the secret with `userdata.get("ANTHROPIC_API_KEY")` and stores it in the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it. Your key never appears in the notebook.

*Running locally instead of Colab?* Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before starting Jupyter.

In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Step 3: Choose the model

This cell stores the model name in a variable called `MODEL_NAME`. Every agent call in this notebook passes `model=MODEL_NAME`, so changing this one line changes the model used across the whole notebook.

We use `claude-haiku-4-5`, a fast and economical model that is more than enough for this lecture. Model names change as new releases arrive, so the comment in the cell links to the models page where you can always find the current list.

In [3]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Step 4: Create the Seed Library files

Before any agent runs, we need something for it to work on. This cell creates a small folder, `/content/seedlibrary_demo/`, with two text files:

| File | What it holds |
| :- | :- |
| `catalog.txt` | Six lines, one per seed, each with a packet count. One count is clearly the largest, so the right answer is easy to check |
| `notes.txt` | A few short lines about swap-day logistics. It is there to make the folder feel real |

Notice what is **not** in these files: nothing about anyone's favourite seed. In a moment you will tell the agent a fact that exists only in your prompt. That detail is what lets us prove later that a session remembers conversation, not files.

The last part of the cell computes a **SHA-256 hash** of `catalog.txt`. A hash is a short fingerprint of a file's exact contents. If even one character changed, the fingerprint would change. We save it now and compare it at the end to prove that nothing we did with sessions touched the files.

In [4]:
import hashlib
from pathlib import Path

LIB_DIR = Path("/content/seedlibrary_demo")
LIB_DIR.mkdir(parents=True, exist_ok=True)

catalog_text = (
    "marigold: 40 packets\n"
    "basil: 12 packets\n"
    "okra: 75 packets\n"
    "sunflower: 28 packets\n"
    "zinnia: 33 packets\n"
    "cosmos: 19 packets\n"
)

notes_text = (
    "Swap day is the first Saturday of every month.\n"
    "Bring seeds in labelled paper envelopes.\n"
    "The sign-in table opens at nine in the morning.\n"
)

(LIB_DIR / "catalog.txt").write_text(catalog_text)
(LIB_DIR / "notes.txt").write_text(notes_text)

CATALOG_HASH_BEFORE = hashlib.sha256((LIB_DIR / "catalog.txt").read_bytes()).hexdigest()
print("Created the Seed Library files.")
print("catalog.txt SHA-256:", CATALOG_HASH_BEFORE)

Created the Seed Library files.
catalog.txt SHA-256: 4c4f310de6ecdd2645f68c8739c087125e7455aa829fc8845d2a400aaadf7da2


## Step 5: Verify what is on disk

Never trust a setup cell without looking at its result. This cell prints the folder tree and then the full contents of each file.

Check two things: the six catalog lines are there, and the largest packet count is unmistakable. Also confirm that the word "favourite" appears nowhere. Everything the agent is about to learn about your favourite seed will come from the prompt alone.

In [5]:
import os

print("Folder tree:")
for folder, subfolders, files in os.walk(LIB_DIR):
    depth = folder.replace(str(LIB_DIR), "").count(os.sep)
    indent = "    " * depth
    print(f"{indent}{os.path.basename(folder)}/")
    for name in sorted(files):
        print(f"{indent}    {name}")

for name in ["catalog.txt", "notes.txt"]:
    print(f"\n--- {name} ---")
    print((LIB_DIR / name).read_text())

Folder tree:
seedlibrary_demo/
    catalog.txt
    notes.txt

--- catalog.txt ---
marigold: 40 packets
basil: 12 packets
okra: 75 packets
sunflower: 28 packets
zinnia: 33 packets
cosmos: 19 packets


--- notes.txt ---
Swap day is the first Saturday of every month.
Bring seeds in labelled paper envelopes.
The sign-in table opens at nine in the morning.



## Step 6: The first call, and where the session ID appears

Now we run the agent. Before you do, hold on to this mental model, because everything else in the notebook builds on it:

> `query()` is **not** a function that returns a value. It returns an **async stream of messages** while the agent loop runs on its own.

You met this loop earlier in the course. The agent reads your prompt, decides to call a tool, receives the tool result, and repeats until it can answer. Each of those steps arrives as a separate message in the stream: assistant messages, tool results (which arrive inside user messages), and finally one **`ResultMessage`** that closes the stream with the final answer.

**The transcript is the saved record of that stream.** The `ResultMessage` is also where the **session ID** lives, in its `session_id` field. It is present on every result, whether the run succeeded or failed.

| Piece of the call | Meaning |
| :- | :- |
| `async for message in query(...)` | Top-level `await` in Colab lets us loop over the stream directly. No event-loop wrapper is needed |
| `prompt=` | Two parts on purpose: a question the agent can answer from the file, and a fact that exists only in this prompt |
| `allowed_tools=["Read"]` | The agent may read files. Nothing else is needed |
| `model=MODEL_NAME` | The model from Step 3 |
| `cwd="/content"` | The working directory. As you will see shortly, the SDK uses it to decide where to save the session |
| `isinstance(message, ResultMessage)` | Picks out the final message from the stream |
| `message.subtype == "success"` | Confirms the run finished normally, so `message.result` holds the answer |

The path in the prompt is `seedlibrary_demo/catalog.txt` because the agent starts in `/content`.

Agents are not scripts, so the wording of the answer will differ from run to run. Look for the right seed and the right count, not identical text.

In [6]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

first_session_id = None

async for message in query(
    prompt=(
        "Read seedlibrary_demo/catalog.txt and tell me which seed has the most "
        "packets and how many. Also, my favourite seed is basil."
    ),
    options=ClaudeAgentOptions(
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        first_session_id = message.session_id
        if message.subtype == "success":
            print(message.result)

print()
print("First session ID:", first_session_id)

Based on the catalog.txt file, **okra** has the most packets with **75 packets**.

(And I see your favorite seed, basil, is in the library with 12 packets available!)

First session ID: 7f286bea-e383-4a7a-a6f2-3c923fb95501


## Step 7: Find the transcript on disk

The SDK wrote that session to disk automatically. The documentation says transcripts live here:

```
~/.claude/projects/<encoded-working-directory>/<session-id>.jsonl
```

The encoded folder name is the absolute working directory with every non-alphanumeric character replaced by a hyphen. If the `CLAUDE_CONFIG_DIR` environment variable is set, it replaces `~/.claude` as the root.

Rather than assume the path, this cell **searches** for it. It works out the projects folder, then looks for a file named after your session ID anywhere inside it. Then it prints:

- the full path, so you can see the folder name that was derived from `/content`,
- the file size,
- the number of lines. A `.jsonl` file holds one JSON record per line.

We also save the transcript's folder as `session_dir`, because Step 12 will count the files in it.

Whether the file is exactly where the documentation says is something this cell lets you confirm for yourself.

In [7]:
from pathlib import Path
import os

config_root = Path(os.environ.get("CLAUDE_CONFIG_DIR") or (Path.home() / ".claude"))
projects_root = config_root / "projects"

print("CLAUDE_CONFIG_DIR is set:", bool(os.environ.get("CLAUDE_CONFIG_DIR")))
print("Projects folder:", projects_root)

matches = list(projects_root.rglob(f"{first_session_id}.jsonl"))
assert matches, "Transcript not found. Check the projects folder above."

transcript_path = matches[0]
session_dir = transcript_path.parent

with open(transcript_path, encoding="utf-8") as f:
    line_count = sum(1 for _ in f)

print()
print("Transcript path:", transcript_path)
print("Folder name:    ", session_dir.name)
print("File size:      ", transcript_path.stat().st_size, "bytes")
print("Lines:          ", line_count)

CLAUDE_CONFIG_DIR is set: False
Projects folder: /root/.claude/projects

Transcript path: /root/.claude/projects/-content/7f286bea-e383-4a7a-a6f2-3c923fb95501.jsonl
Folder name:     -content
File size:       173043 bytes
Lines:           33


## Step 8: Peek inside the transcript (look, don't depend)

Now we open the file. Each line is one JSON record, so this cell reads the file line by line, parses each one with `json.loads`, and prints the record's `type` and its top-level keys.

**Look, don't depend.** The raw transcript format is internal to the SDK and the Claude Code engine beneath it. It can change between releases without warning, so your code should never parse these lines directly. We look at the file once because seeing the real thing makes the idea of a session concrete: it is just a file of records, in order.

In the output, notice how many records there are and how they differ. Do not memorize the keys. The next two steps show the supported way to read a session.

In [8]:
import json

with open(transcript_path, encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        if not line.strip():
            continue
        entry = json.loads(line)
        print(f"Line {number}: type={entry.get('type')!r}")
        print(f"         keys={sorted(entry.keys())}")

Line 1: type='queue-operation'
         keys=['content', 'operation', 'sessionId', 'timestamp', 'type']
Line 2: type='queue-operation'
         keys=['operation', 'sessionId', 'timestamp', 'type']
Line 3: type='user'
         keys=['cwd', 'entrypoint', 'gitBranch', 'isSidechain', 'message', 'parentUuid', 'permissionMode', 'promptId', 'promptSource', 'sessionId', 'timestamp', 'turnOrigin', 'turnPosition', 'type', 'userType', 'uuid', 'version']
Line 4: type='attachment'
         keys=['attachment', 'cwd', 'entrypoint', 'gitBranch', 'isSidechain', 'parentUuid', 'rendered', 'renderedRole', 'sessionId', 'timestamp', 'type', 'userType', 'uuid', 'version']
Line 5: type='attachment'
         keys=['attachment', 'cwd', 'entrypoint', 'gitBranch', 'isSidechain', 'parentUuid', 'rendered', 'renderedRole', 'sessionId', 'timestamp', 'type', 'userType', 'uuid', 'version']
Line 6: type='attachment'
         keys=['attachment', 'cwd', 'entrypoint', 'gitBranch', 'isSidechain', 'parentUuid', 'rendered', '

## Step 9: List and inspect sessions with SDK functions

This is the supported way to find sessions. Both functions are imported from `claude_agent_sdk`, and both are ordinary synchronous functions, so no `await` is needed.

| Function | What it does |
| :- | :- |
| `list_sessions(directory="/content")` | Returns the sessions saved for that working directory, newest first |
| `get_session_info(session_id, directory="/content")` | Returns details for one session, or `None` if it is not found |

Each result is an `SDKSessionInfo` object. This cell prints five of its fields:

| Field | Meaning |
| :- | :- |
| `summary` | A short display title for the session |
| `first_prompt` | The first meaningful prompt in the conversation |
| `cwd` | The working directory the session ran in |
| `file_size` | Size of the transcript file in bytes |
| `created_at` | When the session was created, in milliseconds since the epoch. The cell also converts it to a readable time |

Check that the newest session in the list is the one you just created, and that `cwd` matches what you passed to the agent.

In [9]:
from datetime import datetime, timezone
from claude_agent_sdk import list_sessions, get_session_info

sessions = list_sessions(directory="/content")
print("Sessions found for /content:", len(sessions))
if sessions:
    print("Newest session is the one we just ran:", sessions[0].session_id == first_session_id)

info = get_session_info(first_session_id, directory="/content")

if info is None:
    print("No session info found for", first_session_id)
else:
    print()
    print("summary:     ", info.summary)
    print("first_prompt:", info.first_prompt)
    print("cwd:         ", info.cwd)
    print("file_size:   ", info.file_size, "bytes")
    if info.created_at is not None:
        created = datetime.fromtimestamp(info.created_at / 1000, tz=timezone.utc)
        print("created_at:  ", info.created_at, "ms ->", created.isoformat())

Sessions found for /content: 1
Newest session is the one we just ran: True

summary:      Seedlibrary_demo catalog packet counts
first_prompt: Read seedlibrary_demo/catalog.txt and tell me which seed has the most packets and how many. Also, my favourite seed is basil.
cwd:          /content
file_size:    173043 bytes
created_at:   1792822641507 ms -> 2026-10-24T06:17:21.507000+00:00


## Step 10: Read the conversation back

`get_session_messages(session_id, directory="/content")` reads the transcript and returns the conversation as a list of `SessionMessage` objects in chronological order. If the ID is unknown, it returns an empty list.

Each `SessionMessage` has a `type` (`"user"` or `"assistant"`), a `uuid`, the `session_id`, and a `message` field holding the raw message data. That data can be plain text or a list of content blocks, so the small helper function in this cell turns either form into a one-line description. It labels text blocks, tool calls and tool results, and shortens long text to 80 characters.

Read the output from top to bottom. It is the story of the run in order: your prompt, what the agent decided to do, what came back, and the final answer. Look for where the file read appears in that story.

In [10]:
from claude_agent_sdk import get_session_messages


def describe_content(content):
    if isinstance(content, str):
        return content
    parts = []
    for block in content or []:
        kind = block.get("type")
        if kind == "text":
            parts.append(block.get("text", ""))
        elif kind == "tool_use":
            parts.append(f"[tool call: {block.get('name')}]")
        elif kind == "tool_result":
            parts.append("[tool result]")
        else:
            parts.append(f"[{kind}]")
    return " ".join(parts)


messages = get_session_messages(first_session_id, directory="/content")
print("Messages in the session:", len(messages))
print()

for item in messages:
    content = item.message.get("content") if isinstance(item.message, dict) else None
    snippet = describe_content(content).replace("\n", " ")[:80]
    print(f"{item.type:<10} {item.uuid[:8]}  {snippet}")

Messages in the session: 6

user       a4f61179  Read seedlibrary_demo/catalog.txt and tell me which seed has the most packets an
assistant  dfc58037  [thinking]
assistant  bfd081ea  [tool call: Read]
user       95d889c2  [tool result]
assistant  584e148c  [thinking]
assistant  fc51f385  Based on the catalog.txt file, **okra** has the most packets with **75 packets**


## Step 11: A brand-new call, and what it remembers

Here is the moment this notebook has been building toward. We start a **second, completely independent** `query()` call. Same folder, same options, same model. The only difference is the prompt:

> Which seed did I say was my favourite?

Remember where that fact lives. You told it to the first agent in the prompt. It is not in `catalog.txt`, not in `notes.txt`, and not anywhere on disk in the Seed Library. The first session holds it, and this new call does not have that session.

Two things to do and two things to watch for:

- The cell first counts the transcript files in `session_dir`, so Step 12 can compare before and after.
- Then it runs the call, captures the new session ID, and prints both IDs side by side.

Watch the answer. You are looking for something like "I don't know" or a question back to you. Do not expect exact wording. The agent may also look around the Seed Library folder, because it has the Read tool. Live output varies, and that is a feature of real agents.

In [11]:
jsonl_count_before = len(list(session_dir.glob("*.jsonl")))
second_session_id = None

async for message in query(
    prompt="Which seed did I say was my favourite?",
    options=ClaudeAgentOptions(
        allowed_tools=["Read"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage):
        second_session_id = message.session_id
        if message.subtype == "success":
            print(message.result)

print()
print("First session ID: ", first_session_id)
print("Second session ID:", second_session_id)

I don't have any record of you mentioning a favorite seed in our conversation. This appears to be the first message in our chat, so I don't have any prior context about your seed preferences.

If you'd like to tell me about your favorite seed, I'd be happy to discuss it with you!

First session ID:  7f286bea-e383-4a7a-a6f2-3c923fb95501
Second session ID: b0345f22-21e6-4a85-b7c9-96455dc6eeb8


## Step 12: Check the ground truth

Instead of trusting what the agent said, this cell checks the filesystem, which cannot be argued with. It proves three things:

1. **A new session means a new file.** The number of `.jsonl` files in the transcript folder should be exactly one higher than before the second call.
2. **The two session IDs differ.** Two calls, two sessions.
3. **The files did not change.** We recompute the SHA-256 hash of `catalog.txt` and compare it with the fingerprint from Step 4.

Together these show what a session is: a saved conversation that lives beside your files but is not part of them. The agent answered from the conversation it had, and the second call had no conversation to draw on.

In [12]:
jsonl_count_after = len(list(session_dir.glob("*.jsonl")))
CATALOG_HASH_AFTER = hashlib.sha256((LIB_DIR / "catalog.txt").read_bytes()).hexdigest()

print("Transcript files before the second call:", jsonl_count_before)
print("Transcript files after the second call: ", jsonl_count_after)
print("Increase:", jsonl_count_after - jsonl_count_before)
print()
print("First session ID: ", first_session_id)
print("Second session ID:", second_session_id)
print("IDs are different:", first_session_id != second_session_id)
print()
print("catalog.txt hash before:", CATALOG_HASH_BEFORE)
print("catalog.txt hash after: ", CATALOG_HASH_AFTER)
print("catalog.txt unchanged:  ", CATALOG_HASH_BEFORE == CATALOG_HASH_AFTER)

Transcript files before the second call: 1
Transcript files after the second call:  2
Increase: 1

First session ID:  7f286bea-e383-4a7a-a6f2-3c923fb95501
Second session ID: b0345f22-21e6-4a85-b7c9-96455dc6eeb8
IDs are different: True

catalog.txt hash before: 4c4f310de6ecdd2645f68c8739c087125e7455aa829fc8845d2a400aaadf7da2
catalog.txt hash after:  4c4f310de6ecdd2645f68c8739c087125e7455aa829fc8845d2a400aaadf7da2
catalog.txt unchanged:   True


## What you just saw

- A session is the saved conversation: prompt, tool calls, tool results and responses, in order.
- One `query()` call produced one session, and its ID arrived on the final `ResultMessage`.
- The SDK saved it to disk automatically as a `.jsonl` transcript in a folder derived from the working directory.
- The raw file is internal. `list_sessions()`, `get_session_info()` and `get_session_messages()` are the supported way to read sessions.
- A session stores the conversation, not your files. Your catalog hash never changed.
- A new `query()` call started a new session, so the agent had no memory of what you told the first one. The memory lives in the session, not in the model and not in the files.

**Next:** Lecture 6.2 shows how to capture a session ID and resume that session, so the second agent can remember.